## NN Classification model with selected and full features

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

In [2]:
df = pd.read_csv("data_train.csv")

In [3]:
selected_features = pd.read_csv('selected_features.csv')
selected_features = selected_features.iloc[:, 0].tolist()
len(selected_features)

35

Loaded the final list of selected features produced by correlation-based hierarchical clustering and converted it into a Python list. These 30–40 features form the consistent predictor set for all models in the project.


In [4]:
# Full feature set (drop IDs + targets)
full_feature_cols = [
    c for c in df.columns
    if c not in ["stock_id", "date", "R1M_Usd", "R1M_Usd_C"]
]
len(full_feature_cols)

93

Created the full set of raw predictor features by excluding ID columns and target labels. This defines the baseline feature space before applying correlation-based hierarchical feature selection.


In [5]:
# Training and Validation split in trainging data 2000–2007 for NN tuning
df["date"] = pd.to_datetime(df["date"])

cutoff_date = pd.Timestamp("2006-01-01")

df_train = df[df["date"] <  cutoff_date].copy()   # 2000–2005
df_val   = df[df["date"] >= cutoff_date].copy()   # 2006–2007

print(df_train["date"].min(), df_train["date"].max())
print(df_val["date"].min(),   df_val["date"].max())

1999-12-31 00:00:00 2005-12-31 00:00:00
2006-01-31 00:00:00 2007-12-31 00:00:00


Performed a time-ordered split of the training dataset: 2000–2005 used for fitting and 2006–2007 reserved for validation. This avoids look-ahead bias and provides a proper setup for tuning neural network architectures and hyperparameters.


In [6]:
# X (selected features)
X_train_sel_cls = df_train[selected_features].to_numpy(dtype=np.float32)
X_val_sel_cls   = df_val[selected_features].to_numpy(dtype=np.float32)

# y (binary classification target, e.g. R1M_Usd_C)
y_train_cls = df_train["R1M_Usd_C"].astype(int).to_numpy()
y_val_cls   = df_val["R1M_Usd_C"].astype(int).to_numpy()

input_dim_sel = X_train_sel_cls.shape[1]

The resulting scaled inputs define the neural network’s input dimension for all regression and classification models.

### NN Clasification Model Builder function

Same pattern as regression model but with sigmoid output and binary_crossentropy:

In [7]:
def build_nn_clf_model(input_dim, hidden_layers, l2_strength=0.0, dropout_rate=0.0):
    model = models.Sequential()
    
    # Explicit input layer
    model.add(layers.Input(shape=(input_dim,)))
    
    # Hidden layers
    for units in hidden_layers:
        model.add(
            layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_strength),
            )
        )
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))
    
    # Binary classification output
    model.add(layers.Dense(1, activation="sigmoid"))
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


Defined a flexible neural network builder for regression using customizable hidden layers, L2 regularization, and dropout. The model outputs a single continuous prediction (R1M_Usd) and trains using MSE loss with the Adam optimizer.


In [8]:
## Define a few NN configs to experiment on both full and selected feature sets

nn_clf_configs = [
    {
        "name": "nn_small",
        "hidden_layers": [64, 32],
        "l2_strength": 1e-4,
        "dropout": 0.0,
        "epochs": 30,
        "batch_size": 256,
    },
    {
        "name": "nn_medium_dropout",
        "hidden_layers": [128, 64, 32],
        "l2_strength": 1e-4,
        "dropout": 0.2,
        "epochs": 40,
        "batch_size": 256,
    },
    {
        "name": "nn_deep_strong_reg",
        "hidden_layers": [256, 128, 64],
        "l2_strength": 1e-3,
        "dropout": 0.3,
        "epochs": 50,
        "batch_size": 512,
    },
]


Defined three neural network configurations varying in depth, regularization, dropout, and training parameters. These setups allow systematic experimentation to evaluate model complexity and the impact of feature selection.


In [9]:
results_nn_clf = []

for cfg in nn_clf_configs:
    print(f"\n=== Training {cfg['name']} ===")

    model_clf = build_nn_clf_model(
        input_dim=input_dim_sel,
        hidden_layers=cfg["hidden_layers"],
        l2_strength=cfg["l2_strength"],
        dropout_rate=cfg["dropout"],
    )
    
    history = model_clf.fit(
        X_train_sel_cls, y_train_cls,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_data=(X_val_sel_cls, y_val_cls),
        verbose=0,
    )
    
    # --- Predictions ---
    # probabilities
    y_train_proba = model_clf.predict(X_train_sel_cls).ravel()
    y_val_proba   = model_clf.predict(X_val_sel_cls).ravel()
    
    # class labels (0/1)
    y_train_pred = (y_train_proba >= 0.5).astype(int)
    y_val_pred   = (y_val_proba   >= 0.5).astype(int)
    
    # --- Metrics ---
    acc_train = accuracy_score(y_train_cls, y_train_pred)
    acc_val   = accuracy_score(y_val_cls,   y_val_pred)
    
    auc_train = roc_auc_score(y_train_cls, y_train_proba)
    auc_val   = roc_auc_score(y_val_cls,   y_val_proba)
    
    prec_train = precision_score(y_train_cls, y_train_pred, zero_division=0)
    prec_val   = precision_score(y_val_cls,   y_val_pred, zero_division=0)
    
    rec_train = recall_score(y_train_cls, y_train_pred, zero_division=0)
    rec_val   = recall_score(y_val_cls,   y_val_pred, zero_division=0)
    
    print(f"{cfg['name']} – Train: acc={acc_train:.3f}, AUC={auc_train:.3f}, "
          f"prec={prec_train:.3f}, rec={rec_train:.3f}")
    print(f"{cfg['name']} –  Val : acc={acc_val:.3f}, AUC={auc_val:.3f}, "
          f"prec={prec_val:.3f}, rec={rec_val:.3f}")
    
    results_nn_clf.append({
        "name": cfg["name"],
        "acc_train": acc_train,
        "acc_val":   acc_val,
        "auc_train": auc_train,
        "auc_val":   auc_val,
        "prec_train": prec_train,
        "prec_val":   prec_val,
        "rec_train":  rec_train,
        "rec_val":    rec_val,
    })


=== Training nn_small ===


2025-11-15 22:05:55.390496: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


896/896 [==============================] - 0s 220us/step
nn_small – Train: acc=0.538, AUC=0.563, prec=0.527, rec=0.700
nn_small –  Val : acc=0.508, AUC=0.512, prec=0.503, rec=0.661

=== Training nn_medium_dropout ===
896/896 [==============================] - 0s 235us/step
nn_medium_dropout – Train: acc=0.541, AUC=0.559, prec=0.548, rec=0.434
nn_medium_dropout –  Val : acc=0.510, AUC=0.511, prec=0.509, rec=0.388

=== Training nn_deep_strong_reg ===
896/896 [==============================] - 0s 237us/step
nn_deep_strong_reg – Train: acc=0.533, AUC=0.546, prec=0.531, rec=0.519
nn_deep_strong_reg –  Val : acc=0.505, AUC=0.505, prec=0.501, rec=0.492


Looped over the neural network configurations, training each regression model on the scaled selected features and evaluating MSE, R², and sign accuracy on both train and validation sets. Stored all metrics and training histories to compare architectures and assess the impact of feature selection.


In [ ]:
## Selected Feature
results_nn_clf_df = pd.DataFrame(results_nn_clf)
results_nn_clf_df.sort_values("auc_val", ascending=False)

,name,acc_train,acc_val,auc_train,auc_val,prec_train,prec_val,rec_train,rec_val
0,nn_small,0.538259,0.508232,0.562765,0.512301,0.526725,0.503349,0.700397,0.660503
1,nn_medium_dropout,0.540610,0.510464,0.559322,0.510565,0.547861,0.508714,0.433859,0.387866
2,nn_deep_strong_reg,0.533178,0.504918,0.545900,0.504550,0.531147,0.501074,0.518862,0.491985


#### Full Feature

In [11]:
# ==== FULL feature matrices for classification ====

X_train_full_cls = df_train[full_feature_cols].to_numpy(dtype=np.float32)
X_val_full_cls   = df_val[full_feature_cols].to_numpy(dtype=np.float32)

y_train_cls = df_train["R1M_Usd_C"].astype(int).to_numpy()
y_val_cls   = df_val["R1M_Usd_C"].astype(int).to_numpy()

input_dim_full = X_train_full_cls.shape[1]

In [12]:
results_nn_clf_full = []

for cfg in nn_clf_configs:
    print(f"\n=== Training {cfg['name']} (FULL) ===")

    model_clf_full = build_nn_clf_model(
        input_dim=input_dim_full,
        hidden_layers=cfg["hidden_layers"],
        l2_strength=cfg["l2_strength"],
        dropout_rate=cfg["dropout"],
    )
    
    history = model_clf_full.fit(
        X_train_full_cls, y_train_cls,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_data=(X_val_full_cls, y_val_cls),
        verbose=0,
    )
    
    # --- Predictions ---
    y_train_proba = model_clf_full.predict(X_train_full_cls).ravel()
    y_val_proba   = model_clf_full.predict(X_val_full_cls).ravel()
    
    y_train_pred = (y_train_proba >= 0.5).astype(int)
    y_val_pred   = (y_val_proba   >= 0.5).astype(int)
    
    # --- Metrics ---
    acc_train = accuracy_score(y_train_cls, y_train_pred)
    acc_val   = accuracy_score(y_val_cls,   y_val_pred)
    
    auc_train = roc_auc_score(y_train_cls, y_train_proba)
    auc_val   = roc_auc_score(y_val_cls,   y_val_proba)
    
    prec_train = precision_score(y_train_cls, y_train_pred, zero_division=0)
    prec_val   = precision_score(y_val_cls,   y_val_pred, zero_division=0)
    
    rec_train = recall_score(y_train_cls, y_train_pred, zero_division=0)
    rec_val   = recall_score(y_val_cls,   y_val_pred, zero_division=0)
    
    print(f"{cfg['name']} – Train: acc={acc_train:.3f}, AUC={auc_train:.3f}, "
          f"prec={prec_train:.3f}, rec={rec_train:.3f}")
    print(f"{cfg['name']} –  Val : acc={acc_val:.3f}, AUC={auc_val:.3f}, "
          f"prec={prec_val:.3f}, rec={rec_val:.3f}")
    
    results_nn_clf_full.append({
        "name": cfg["name"],
        "acc_train": acc_train,
        "acc_val":   acc_val,
        "auc_train": auc_train,
        "auc_val":   auc_val,
        "prec_train": prec_train,
        "prec_val":   prec_val,
        "rec_train":  rec_train,
        "rec_val":    rec_val,
    })



=== Training nn_small (FULL) ===
896/896 [==============================] - 0s 198us/step
nn_small – Train: acc=0.543, AUC=0.562, prec=0.550, rec=0.445
nn_small –  Val : acc=0.509, AUC=0.505, prec=0.506, rec=0.432

=== Training nn_medium_dropout (FULL) ===
896/896 [==============================] - 0s 235us/step
nn_medium_dropout – Train: acc=0.539, AUC=0.557, prec=0.534, rec=0.563
nn_medium_dropout –  Val : acc=0.511, AUC=0.515, prec=0.506, rec=0.529

=== Training nn_deep_strong_reg (FULL) ===
896/896 [==============================] - 0s 305us/step
nn_deep_strong_reg – Train: acc=0.534, AUC=0.548, prec=0.529, rec=0.567
nn_deep_strong_reg –  Val : acc=0.502, AUC=0.505, prec=0.499, rec=0.543


In [ ]:
# FOr FULL feature metrics DataFrame to inspect
results_nn_clf_full_df = pd.DataFrame(results_nn_clf_full)
results_nn_clf_full_df.sort_values("auc_val", ascending=False)

,name,acc_train,acc_val,auc_train,auc_val,prec_train,prec_val,rec_train,rec_val
1,nn_medium_dropout,0.538720,0.510534,0.557330,0.514930,0.534150,0.506422,0.563099,0.529457
0,nn_small,0.543032,0.509243,0.562200,0.505337,0.549795,0.506348,0.445387,0.431805
2,nn_deep_strong_reg,0.533627,0.502337,0.548269,0.504779,0.528786,0.498581,0.567211,0.543448


In [14]:
# Rename model types if you really want "Classification" wording
results_nn_clf_full_df["model_type"] = "NN_Classification_Full"
results_nn_clf_df["model_type"]      = "NN_Classification_Selected"

# Merge
results_nn_clf_all = pd.concat(
    [results_nn_clf_full_df, results_nn_clf_df],
    ignore_index=True
)

# Sort: first by model_type block, then by auc_val within each block
results_nn_clf_all = results_nn_clf_all.sort_values(
    ["model_type", "auc_val"],
    ascending=[True, False]
)

# Make model_type the first column
cols = ["model_type"] + [c for c in results_nn_clf_all.columns if c != "model_type"]
results_nn_clf_all = results_nn_clf_all[cols]

results_nn_clf_all


,model_type,name,acc_train,acc_val,auc_train,auc_val,prec_train,prec_val,rec_train,rec_val
1,NN_Classification_Full,nn_medium_dropout,0.538720,0.510534,0.557330,0.514930,0.534150,0.506422,0.563099,0.529457
0,NN_Classification_Full,nn_small,0.543032,0.509243,0.562200,0.505337,0.549795,0.506348,0.445387,0.431805
2,NN_Classification_Full,nn_deep_strong_reg,0.533627,0.502337,0.548269,0.504779,0.528786,0.498581,0.567211,0.543448
3,NN_Classification_Selected,nn_small,0.538259,0.508232,0.562765,0.512301,0.526725,0.503349,0.700397,0.660503
4,NN_Classification_Selected,nn_medium_dropout,0.540610,0.510464,0.559322,0.510565,0.547861,0.508714,0.433859,0.387866
5,NN_Classification_Selected,nn_deep_strong_reg,0.533178,0.504918,0.545900,0.504550,0.531147,0.501074,0.518862,0.491985


## Backtest - Rebalancing Loop

In [15]:
data = pd.read_csv("data_backtest.csv").copy()
data["date"] = pd.to_datetime(data["date"])
data["ym"]   = data["date"].dt.to_period("M")

In [16]:
backtest_start = "2008-01-01"
backtest_end   = "2018-12-31"

LOOKBACK_MONTHS = [12, 24, 36, 48, 60]   # lookback periods to try

In [17]:
## Function to get month window
def get_month_window(df, current_ym, lookback):
    start_ym = current_ym - lookback    # period math
    mask = (df["ym"] > start_ym) & (df["ym"] < current_ym)
    return df[mask].copy()

In [18]:
def run_backtest_nn_clf(df, features, lookback_months, cfg):
    """
    Rolling monthly backtest using NN *classification* model.

    - df: full panel data with columns: 'date', 'ym', features, 'R1M_Usd', 'R1M_Usd_C'
    - features: list of feature column names
    - lookback_months: int, number of months in training window
    - cfg: dict with keys: hidden_layers, l2_strength, dropout, epochs, batch_size

    Returns: pd.Series of monthly long-short PnL.
    """

    monthly_returns = []
    month_index = []

    # Year-months within backtest period
    months = (
        df.loc[(df["date"] >= backtest_start) & (df["date"] <= backtest_end), "ym"]
          .sort_values()
          .unique()
    )

    for ym in months:
        # 1) Training window: previous L months
        train_df = get_month_window(df, ym, lookback_months)
        # 2) Test month: current ym
        test_df  = df[df["ym"] == ym]

        # Guard: enough data
        if len(train_df) < 500 or len(test_df) == 0:
            continue

        # 3) Prepare X, y (classification label)
        X_train = train_df[features].values.astype("float32")
        y_train = train_df["R1M_Usd_C"].values.astype("float32")   # 0/1 classification

        X_test  = test_df[features].values.astype("float32")
        # y_test_cls = test_df["R1M_Usd_C"].values.astype("float32")  # not needed for trading

        # 4) Scale features (fit on train, transform both)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        # 5) Build & train NN CLASSIFICATION model (fresh each month)
        model = build_nn_clf_model(
            input_dim=X_train_scaled.shape[1],
            hidden_layers=cfg["hidden_layers"],
            l2_strength=cfg["l2_strength"],
            dropout_rate=cfg["dropout"]
        )

        model.fit(
            X_train_scaled,
            y_train,
            epochs=cfg["epochs"],
            batch_size=cfg["batch_size"],
            verbose=0  # no validation here; we don't peek into test month
        )

        # 6) Predict probabilities for next month
        #    Use P(y=1) as ranking signal
        prob_pos = model.predict(X_test_scaled, verbose=0).flatten()

        # 7) Build long-short portfolio based on prob_pos
        test_df = test_df.copy()
        test_df["score"] = prob_pos   # higher score = more likely positive return

        test_df = test_df.sort_values("score")
        n = len(test_df)
        if n < 10:
            continue

        # bottom 20% = short, top 20% = long
        long_df  = test_df.iloc[int(0.8 * n):]   # top 20% scores
        short_df = test_df.iloc[:int(0.2 * n)]   # bottom 20% scores

        w_long  = 1.0 / len(long_df)
        w_short = -1.0 / len(short_df)

        # Realized returns use R1M_Usd
        long_ret  = (long_df["R1M_Usd"] * w_long).sum()
        short_ret = (short_df["R1M_Usd"] * w_short).sum()

        monthly_returns.append(long_ret + short_ret)
        # Month-end timestamp index
        month_index.append(ym.to_timestamp("M"))

    return pd.Series(monthly_returns, index=month_index)

In [19]:
## BEst config from NN regression experiments

best_nn_cfg =  {
        "name": "nn_deep_strong_reg",
        "hidden_layers": [256, 128, 64],
        "l2_strength": 1e-3,
        "dropout": 0.3,
        "epochs": 50,
        "batch_size": 512,
    }

In [20]:
results_storage_selected = {}

for L in LOOKBACK_MONTHS:
    print(f"Running NN-REG Backtest with Lookback {L} months...")
    pnl_series_nn = run_backtest_nn_clf(data, selected_features, L, best_nn_cfg)
    results_storage_selected[f"NNreg_Lookback_{L}"] = pnl_series_nn

Running NN-REG Backtest with Lookback 12 months...
Running NN-REG Backtest with Lookback 24 months...
Running NN-REG Backtest with Lookback 36 months...
Running NN-REG Backtest with Lookback 48 months...
Running NN-REG Backtest with Lookback 60 months...


In [21]:
results_storage_selected

{'NNreg_Lookback_12': 2008-01-31    0.025883
 2008-02-29   -0.023562
 2008-03-31    0.000126
 2008-04-30    0.027364
 2008-05-31    0.078939
                 ...   
 2018-07-31    0.126117
 2018-08-31    0.003681
 2018-09-30    0.012938
 2018-10-31    0.015739
 2018-11-30    0.027046
 Length: 131, dtype: float64,
 'NNreg_Lookback_24': 2008-01-31    0.035133
 2008-02-29   -0.018253
 2008-03-31    0.006258
 2008-04-30    0.010752
 2008-05-31    0.057748
                 ...   
 2018-07-31    0.014479
 2018-08-31    0.011084
 2018-09-30   -0.011775
 2018-10-31    0.021463
 2018-11-30    0.016249
 Length: 131, dtype: float64,
 'NNreg_Lookback_36': 2008-01-31    0.044072
 2008-02-29   -0.013589
 2008-03-31    0.030911
 2008-04-30    0.063076
 2008-05-31    0.054203
                 ...   
 2018-07-31    0.017317
 2018-08-31    0.006532
 2018-09-30    0.000618
 2018-10-31    0.031810
 2018-11-30   -0.006349
 Length: 131, dtype: float64,
 'NNreg_Lookback_48': 2008-01-31    0.060369
 2008-02-2

In [22]:
results_storage_all = {}

for L in LOOKBACK_MONTHS:
    print(f"Running NN-REG Backtest with Lookback {L} months...")
    pnl_series_nn = run_backtest_nn_clf(data, full_feature_cols, L, best_nn_cfg)
    results_storage_all[f"NNreg_Lookback_{L}"] = pnl_series_nn

Running NN-REG Backtest with Lookback 12 months...
Running NN-REG Backtest with Lookback 24 months...
Running NN-REG Backtest with Lookback 36 months...
Running NN-REG Backtest with Lookback 48 months...
Running NN-REG Backtest with Lookback 60 months...


In [23]:
results_storage_all

{'NNreg_Lookback_12': 2008-01-31    0.023920
 2008-02-29   -0.022389
 2008-03-31    0.002897
 2008-04-30    0.014876
 2008-05-31    0.080656
                 ...   
 2018-07-31   -0.006337
 2018-08-31    0.016456
 2018-09-30   -0.004630
 2018-10-31    0.030837
 2018-11-30    0.022133
 Length: 131, dtype: float64,
 'NNreg_Lookback_24': 2008-01-31    0.037974
 2008-02-29   -0.025197
 2008-03-31    0.009048
 2008-04-30    0.039445
 2008-05-31    0.070608
                 ...   
 2018-07-31    0.017538
 2018-08-31    0.022339
 2018-09-30    0.008356
 2018-10-31    0.025693
 2018-11-30    0.015841
 Length: 131, dtype: float64,
 'NNreg_Lookback_36': 2008-01-31    0.041300
 2008-02-29   -0.031123
 2008-03-31    0.047137
 2008-04-30    0.036175
 2008-05-31    0.049884
                 ...   
 2018-07-31   -0.109210
 2018-08-31    0.010220
 2018-09-30    0.009442
 2018-10-31    0.024924
 2018-11-30   -0.016662
 Length: 131, dtype: float64,
 'NNreg_Lookback_48': 2008-01-31    0.054945
 2008-02-2

In [24]:
## Equally Weighted Portfolio

def build_ew_market_series(df):
    # Restrict to backtest period
    df_bt = df[(df["date"] >= backtest_start) & (df["date"] <= backtest_end)].copy()
    df_bt["ym"] = df_bt["date"].dt.to_period("M")

    # Equal-weight average of R1M_Usd across all stocks each month
    ew = df_bt.groupby("ym")["R1M_Usd"].mean()
    ew.index = ew.index.to_timestamp("M")   # month-end timestamps
    return ew

ew_market = build_ew_market_series(data)

In [25]:
## Performace Metrics

def perf_stats(pnl: pd.Series, ew_market: pd.Series):
    """
    pnl: monthly return series of a strategy (or EW itself)
    ew_market: monthly EW market returns
    """
    # Align with EW
    aligned = pd.concat([pnl.astype(float), ew_market.astype(float)], axis=1, join="inner")
    aligned.columns = ["strategy", "ew"]
    r = aligned["strategy"].dropna()
    if len(r) < 2:
        return None

    # Basic stats
    cum = (1 + r).prod()
    ann_ret = cum**(12 / len(r)) - 1

    ann_vol = r.std() * (12**0.5)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else float("nan")

    # Max drawdown
    cum_curve = (1 + r).cumprod()
    peak = cum_curve.cummax()
    dd = (cum_curve / peak) - 1
    max_dd = dd.min()

    # Hit rate
    hit_rate = (r > 0).mean()

    # Correlation to EW
    corr_ew = aligned["strategy"].corr(aligned["ew"])

    return {
        "n_months": len(r),
        "ann_ret": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "hit_rate": hit_rate,
        "corr_to_ew": corr_ew,
    }

In [26]:
## Comapring the SELECTED feature set results vs Equally Weighted Portfolio

all_strategies = {}

# NN classification
for name, series in results_storage_selected.items():
    all_strategies[name] = series

rows = []

# EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df

,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
NNreg_Lookback_60,131,0.040264,0.108084,0.372527,-0.229331,0.549618,0.173691
NNreg_Lookback_48,131,0.001935,0.101554,0.019056,-0.258500,0.480916,-0.076953
NNreg_Lookback_36,131,-0.003258,0.111940,-0.029105,-0.379242,0.488550,0.021814
NNreg_Lookback_12,131,-0.004676,0.114193,-0.040950,-0.344606,0.557252,-0.303155
NNreg_Lookback_24,131,-0.048321,0.111917,-0.431755,-0.493771,0.503817,-0.348756


In [27]:
## Comapring the FULL feature set results vs Equally Weighted Portfolio

all_strategies = {}

# NN classification
for name, series in results_storage_all.items():
    all_strategies[name] = series

rows = []

# EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df

,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
NNreg_Lookback_60,131,0.021975,0.100558,0.218528,-0.234713,0.526718,0.082701
NNreg_Lookback_12,131,-0.012370,0.109510,-0.112957,-0.355994,0.488550,-0.418696
NNreg_Lookback_48,131,-0.014717,0.114784,-0.128211,-0.334302,0.511450,-0.089188
NNreg_Lookback_24,131,-0.024389,0.120940,-0.201662,-0.418993,0.549618,-0.295306
NNreg_Lookback_36,131,-0.050163,0.105499,-0.475478,-0.517286,0.465649,-0.289205


### NN Classification did not perform any betetr that EW Benchmark